In [1]:
from models.audio_llm import AudioLLMModel
from types import SimpleNamespace
import torch
from dataset.audio_dataset import AudioChatDataset, collate_fn
from torch.utils.data import DataLoader
import functools
from transformers import AutoProcessor
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
# config
# python train.py model.lm_model_name=google/gemma-2-9b-it model.encoder_model_name=syvai/hviske-v2 training.batch_size=32 dataset.name=coral model.load_in_4bit=true wandb.enabled=true training.resume_from_checkpoint=true training.resume_checkpoint_path=/work3/s224233/shared/multimodal-alignment/outputs/2025-02-14/13-36-39/checkpoints/checkpoint_latest.pt

config = SimpleNamespace(
    model=SimpleNamespace(
        lm_model_name="google/gemma-2-9b-it",
        encoder_model_name="syvai/hviske-v2",
        downsample_factor=10,
        load_in_4bit=True,
    ),
    training=SimpleNamespace(kl_loss=True),
)

model = AudioLLMModel(config)

checkpoint = torch.load(
    "/zhome/5c/8/186818/multimodal-alignment/outputs/2025-02-15/12-42-21/checkpoints/checkpoint_latest.pt"
)

model.projector.load_state_dict(checkpoint["Projector"])

You are attempting to use Flash Attention 2.0 with a model not initialized on GPU. Make sure to move the model to GPU after initializing it on CPU with `model.to('cuda')`.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

<All keys matched successfully>

In [3]:
tokenizer = model.tokenizer
processor = AutoProcessor.from_pretrained("syvai/hviske-v2")

In [4]:
test_dataset = AudioChatDataset(
    lm_model="google/gemma-2-9b-it",
    dataset="coral",
    tokenizer=tokenizer,
    processor=processor,
    split="test",
)

In [5]:
n_samples = 20

In [7]:
# Process entire batch at once
def get_user_prompts_batch(ids_batch, model_name, tokenizer):
    token_id = tokenizer.bos_token_id
    if model_name == "gemma":
        token_id = 106  # <|start_of_turn|>

    # Initialize list to store prompts and their masks
    prompts = []
    masks = []

    # Process each sequence in the batch
    for ids in ids_batch:
        # Find positions of BOS tokens
        bos_positions = (ids == token_id).nonzero(as_tuple=True)[0]
        # Get the last BOS token position (start of assistant's response)
        last_bos = bos_positions[-1].item()
        # Get everything before the assistant's response
        prompt = ids[: last_bos + 3]
        prompts.append(prompt)
        masks.append(torch.ones_like(prompt))

    # Pad sequences to same length
    max_len = max(len(p) for p in prompts)
    padded_prompts = []
    padded_masks = []

    for prompt, mask in zip(prompts, masks):
        pad_len = max_len - len(prompt)
        padded_prompts.append(
            torch.nn.functional.pad(prompt, (0, pad_len), value=tokenizer.pad_token_id)
        )
        padded_masks.append(torch.nn.functional.pad(mask, (0, pad_len), value=0))

    return torch.stack(padded_prompts), torch.stack(padded_masks)


def get_ground_truths_batch(ids_batch, model_name, tokenizer):
    begin_of_turn = tokenizer.bos_token_id
    end_of_turn = tokenizer.eos_token_id
    if model_name == "gemma":
        begin_of_turn = 106  # <|start_of_turn|>
        end_of_turn = 107  # <|end_of_turn|>

    ground_truths = []

    for ids in ids_batch:
        # Find positions of BOS and EOS tokens
        bos_positions = (ids == begin_of_turn).nonzero(as_tuple=True)[0]
        eos_positions = (ids == end_of_turn).nonzero(as_tuple=True)[0]

        # Get the last BOS token (start of assistant response)
        last_bos = bos_positions[-1].item()

        # Find the first EOS after last BOS
        eos_after_bos = eos_positions[eos_positions > last_bos]
        response_end = eos_after_bos[0].item() if len(eos_after_bos) > 0 else len(ids)

        # Extract and decode the assistant's response
        response_ids = ids[last_bos + 3 : response_end]
        ground_truths.append(tokenizer.decode(response_ids, skip_special_tokens=True))

    return ground_truths


# Perform batch generation
def generate_batch(model, batch, model_name, tokenizer, max_new_tokens=100):
    # Get padded prompts and attention masks for the batch
    user_prompts, user_prompt_masks = get_user_prompts_batch(
        batch["input_ids"], model_name, tokenizer
    )
    ground_truths = get_ground_truths_batch(batch["input_ids"], model_name, tokenizer)

    # Move everything to GPU
    user_prompts = user_prompts.to("cuda")
    user_prompt_masks = user_prompt_masks.to("cuda")
    audio_features = batch["audio_features"].to("cuda", torch.bfloat16)
    insertion_indices = [indices.to("cuda") for indices in batch["insertion_indices"]]

    # Generate completions for the entire batch
    with torch.inference_mode():
        outputs = model.generate(
            input_ids=user_prompts,
            audio_features=audio_features,
            insertion_indices=insertion_indices,
            attention_mask=user_prompt_masks,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    # Decode all generated sequences
    generated_texts = [
        tokenizer.decode(output.tolist(), skip_special_tokens=True)
        for output in outputs
    ]

    return generated_texts, ground_truths


# Main execution
model.eval()

test_loader = DataLoader(
    test_dataset,
    batch_size=n_samples,
    collate_fn=functools.partial(collate_fn, tokenizer=tokenizer),
    shuffle=True,
)
model_name = config.model.lm_model_name.split("/")[-1].split("-")[0].lower()
# get one batch
batch = next(iter(test_loader))
generated_texts, ground_truths = generate_batch(model, batch, model_name, tokenizer)

# Print results
for i, (generated, truth) in enumerate(zip(generated_texts, ground_truths)):
    print(f"\nSample {i + 1}")
    print(f"Generated: {generated}")
    print(f"Ground truth: {truth}")
    print("-" * 80)

The 'batch_size' attribute of HybridCache is deprecated and will be removed in v4.49. Use the more precisely named 'self.max_batch_size' attribute instead.



Sample 1
Generated: området omfatter både bakkeøer og smeltevandsletter og fliede steder er der præget af sandflugter og der er flere markante indlandsklitter 

Ground truth: området omfatter både bakkeøer og smeltevandssletter og flere steder er der præg af sandflugt og der er flere markante indlandsklitter
--------------------------------------------------------------------------------

Sample 2
Generated: med sin tusindårige historie har den været både kroningssted og begravelsesplads for de polske konger 

Ground truth: med sin tusindårige historie har den været både kroningssted og begravelsesplads for de polske konger
--------------------------------------------------------------------------------

Sample 3
Generated: miss universe 1956 var den femte årlige miss universe konkurrence 

Ground truth: miss universe 1956 var den femte årlige miss universe konkurrence
--------------------------------------------------------------------------------

Sample 4
Generated: de nærmeste afh

In [15]:
import jiwer

references = ground_truths
hypotheses = generated_texts

transformation = jiwer.Compose(
    [
        jiwer.ToLowerCase(),
        jiwer.RemoveMultipleSpaces(),
        jiwer.Strip(),
        jiwer.RemovePunctuation(),
    ]
)

wer = jiwer.wer(
    references,
    hypotheses,
)

cer = jiwer.cer(
    references,
    hypotheses,
)

In [16]:
wer, cer

(0.15725806451612903, 0.0707070707070707)

In [9]:
references

['området omfatter både bakkeøer og smeltevandssletter og flere steder er der præg af sandflugt og der er flere markante indlandsklitter',
 'med sin tusindårige historie har den været både kroningssted og begravelsesplads for de polske konger',
 'miss universe 1956 var den femte årlige miss universe konkurrence',
 'de nærmeste arvinger tilhører en anden linje',
 'her tjente vinduet nok så meget til at skaffe luft som til at give lys',
 'den seneste rekord der blev opnået indendørs var i 2004',
 'han er bedst kendt for sit mangeårige samarbejde med bryan adams',
 'den store mission fik et dårligt ry og de sidste ti programmer fra usa blev sendt tirsdag aften',
 'flaget har en rød trekant i den inderste kant og to sorte fempunkede stjerner i det gule felt',
 'dyrene er født ind i menneskenes verden',
 'hun fik sin inspiration til dommergerningen efter sin søster trine k',
 'stade de france er et stadionanlæg i paris',
 'ljungbyhed er en landsby i klippans kommune i skåne',
 'i vøluspa fu

In [11]:
generated_texts

['området omfatter både bakkeøer og smeltevandsletter og fliede steder er der præget af sandflugter og der er flere markante indlandsklitter \n',
 'med sin tusindårige historie har den været både kroningssted og begravelsesplads for de polske konger \n',
 'miss universe 1956 var den femte årlige miss universe konkurrence \n',
 'de nærmeste afhængiger tilhører en anden linje \n',
 'her kendte vindene nok så meget til at skaffe luft som til at skaffe lys \n',
 'den seneste rekord der blev opnået indendørs var i 2004 \n',
 'han er bedst kendt for sine mange år i samarbejde med brand adams \n',
 'den store mission fik et dårligt ryg og de sidste 10 programmer fra usa blev sendt tirsdag aften \n',
 'flaget har en rød trekant i den inderste kant og to sorte fempunktede stjerner i det gule felt \n',
 'dyrene er født af i menneskenes verden \n',
 'hun fik sin inspiration til dommergangan efter sin søster trine k \n',
 'stade de france er et stadionanlæg i paris \n',
 'lyngby-hedegård er en lan

In [1]:
from models.audio_llm import AudioLLMModel
from types import SimpleNamespace
import torch
from dataset.audio_dataset import AudioChatDataset, collate_fn
from torch.utils.data import DataLoader, Subset
import functools
from transformers import AutoProcessor
import os
import jiwer
from tqdm import tqdm
import numpy as np

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Config
config = SimpleNamespace(
    model=SimpleNamespace(
        lm_model_name="google/gemma-2-9b-it",
        encoder_model_name="syvai/hviske-v2",
        downsample_factor=10,
        load_in_4bit=True,
    ),
    training=SimpleNamespace(kl_loss=True),
)


def get_user_prompts_batch(ids_batch, model_name, tokenizer):
    token_id = tokenizer.bos_token_id
    if model_name == "gemma":
        token_id = 106  # <|start_of_turn|>

    prompts = []
    masks = []

    for ids in ids_batch:
        bos_positions = (ids == token_id).nonzero(as_tuple=True)[0]
        last_bos = bos_positions[-1].item()
        prompt = ids[: last_bos + 3]
        prompts.append(prompt)
        masks.append(torch.ones_like(prompt))

    max_len = max(len(p) for p in prompts)
    padded_prompts = []
    padded_masks = []

    for prompt, mask in zip(prompts, masks):
        pad_len = max_len - len(prompt)
        padded_prompts.append(
            torch.nn.functional.pad(prompt, (0, pad_len), value=tokenizer.pad_token_id)
        )
        padded_masks.append(torch.nn.functional.pad(mask, (0, pad_len), value=0))

    return torch.stack(padded_prompts), torch.stack(padded_masks)


def get_ground_truths_batch(ids_batch, model_name, tokenizer):
    begin_of_turn = tokenizer.bos_token_id
    end_of_turn = tokenizer.eos_token_id
    if model_name == "gemma":
        begin_of_turn = 106  # <|start_of_turn|>
        end_of_turn = 107  # <|end_of_turn|>

    ground_truths = []

    for ids in ids_batch:
        bos_positions = (ids == begin_of_turn).nonzero(as_tuple=True)[0]
        eos_positions = (ids == end_of_turn).nonzero(as_tuple=True)[0]
        last_bos = bos_positions[-1].item()
        eos_after_bos = eos_positions[eos_positions > last_bos]
        response_end = eos_after_bos[0].item() if len(eos_after_bos) > 0 else len(ids)
        response_ids = ids[last_bos + 3 : response_end]
        ground_truths.append(tokenizer.decode(response_ids, skip_special_tokens=True))

    return ground_truths


def generate_batch(model, batch, model_name, tokenizer, max_new_tokens=100):
    user_prompts, user_prompt_masks = get_user_prompts_batch(
        batch["input_ids"], model_name, tokenizer
    )
    ground_truths = get_ground_truths_batch(batch["input_ids"], model_name, tokenizer)

    user_prompts = user_prompts.to("cuda")
    user_prompt_masks = user_prompt_masks.to("cuda")
    audio_features = batch["audio_features"].to("cuda", torch.bfloat16)
    insertion_indices = [indices.to("cuda") for indices in batch["insertion_indices"]]

    with torch.inference_mode():
        outputs = model.generate(
            input_ids=user_prompts,
            audio_features=audio_features,
            insertion_indices=insertion_indices,
            attention_mask=user_prompt_masks,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    generated_texts = [
        tokenizer.decode(output.tolist(), skip_special_tokens=True)
        for output in outputs
    ]

    return generated_texts, ground_truths


def calculate_metrics(hypotheses, references):
    wer = jiwer.wer(
        references,
        hypotheses,
    )
    cer = jiwer.cer(
        references,
        hypotheses,
    )

    return wer, cer


# Initialize model
model = AudioLLMModel(config)
checkpoint = torch.load(
    "/zhome/5c/8/186818/multimodal-alignment/outputs/2025-02-15/12-42-21/checkpoints/checkpoint_latest.pt"
)
model.projector.load_state_dict(checkpoint["Projector"])
model.eval()

# Initialize tokenizer and processor
tokenizer = model.tokenizer
processor = AutoProcessor.from_pretrained("syvai/hviske-v2")

# Create test dataset
full_test_dataset = AudioChatDataset(
    lm_model="google/gemma-2-9b-it",
    dataset="coral",
    tokenizer=tokenizer,
    processor=processor,
    split="test",
)

# Create a subset of 64 examples
subset_indices = list(range(min(128, len(full_test_dataset))))
test_dataset = Subset(full_test_dataset, subset_indices)

batch_size = 128
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=functools.partial(collate_fn, tokenizer=tokenizer),
    shuffle=False,
)

model_name = config.model.lm_model_name.split("/")[-1].split("-")[0].lower()

# Lists to store all generated texts and ground truths
all_generated = []
all_ground_truths = []

# Process all batches
print("Generating predictions...")
for batch in tqdm(test_loader):
    generated_texts, ground_truths = generate_batch(model, batch, model_name, tokenizer)
    all_generated.extend(generated_texts)
    all_ground_truths.extend(ground_truths)

# Calculate metrics
print("\nCalculating WER and CER...")
wer, cer = calculate_metrics(all_generated, all_ground_truths)

# Print results
print(f"\nResults over {len(all_generated)} samples:")
print(f"Word Error Rate (WER): {wer:.4f}")
print(f"Character Error Rate (CER): {cer:.4f}")

# Save some example predictions
print("\nSaving example predictions...")
num_examples = min(5, len(all_generated))
with open("prediction_examples.txt", "w") as f:
    for i in range(num_examples):
        f.write(f"\nExample {i + 1}\n")
        f.write(f"Generated: {all_generated[i]}\n")
        f.write(f"Ground truth: {all_ground_truths[i]}\n")
        f.write("-" * 80 + "\n")

You are attempting to use Flash Attention 2.0 with a model not initialized on GPU. Make sure to move the model to GPU after initializing it on CPU with `model.to('cuda')`.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Generating predictions...


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:17<00:00, 17.25s/it]


Calculating WER and CER...

Results over 128 samples:
Word Error Rate (WER): 0.1503
Character Error Rate (CER): 0.0606

Saving example predictions...
